# **Section 4: Data Privacy**

## **Introduction**
In this section, we will explore data privacy in federated learning. We will integrate Differential Privacy (DP) into the training process using the Flower framework to enhance security and protect user data.

---

## **Step 1: Import Required Libraries**


In [ ]:
pip install -r requirements.txt

In [ ]:
from flwr.client.mod import adaptiveclipping_mod
from flwr.server.strategy import (
    DifferentialPrivacyClientSideAdaptiveClipping,
    FedAvg,
)

from utils4 import *

---

## **Step 2: Load and Preprocess the MNIST Dataset**
We will use Flower Datasets to load and preprocess the MNIST dataset in a federated learning setting.

### **Load Federated Dataset**


In [ ]:
def load_data(partition_id):
    fds = FederatedDataset(dataset="mnist", partitioners={"train": 10})
    partition = fds.load_partition(partition_id)

    traintest = partition.train_test_split(test_size=0.2, seed=42)
    traintest = traintest.with_transform(normalize)
    trainset, testset = traintest["train"], traintest["test"]

    trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
    testloader = DataLoader(testset, batch_size=64)
    return trainloader, testloader

---

## **Step 3: Implement Client-Side Training and Evaluation**
We will implement a federated learning client that incorporates Differential Privacy.

### **Define Flower Client**


In [ ]:
class FlowerClient(NumPyClient):
    def __init__(self, net, trainloader, testloader):
        self.net = net
        self.trainloader = trainloader
        self.testloader = testloader

    def fit(self, parameters, config):
        set_weights(self.net, parameters)
        train_model(self.net, self.trainloader)
        return get_weights(self.net), len(self.trainloader), {}

    def evaluate(self, parameters, config):
        set_weights(self.net, parameters)
        loss, accuracy = evaluate_model(self.net, self.testloader)
        return loss, len(self.testloader), {"accuracy": accuracy}

### **Create the Client Function and Client App**


In [ ]:
def client_fn(context: Context) -> Client:
    net = SimpleModel()
    partition_id = int(context.node_config["partition-id"])
    trainloader, testloader = load_data(partition_id=partition_id)
    return FlowerClient(net, trainloader, testloader).to_client()

client = ClientApp(
    client_fn,
    mods=[adaptiveclipping_mod],  # Apply Differential Privacy modifiers
)

---

## **Step 4: Implement Server-Side Strategy with Differential Privacy**
The server will implement Differential Privacy through adaptive clipping and noise addition.

### **Define Federated Averaging with Differential Privacy**


In [ ]:
net = SimpleModel()
params = ndarrays_to_parameters(get_weights(net))

def server_fn(context: Context):
    fedavg_without_dp = FedAvg(
        fraction_fit=0.6,
        fraction_evaluate=1.0,
        initial_parameters=params,
    )
    fedavg_with_dp = DifferentialPrivacyClientSideAdaptiveClipping(
        fedavg_without_dp,
        noise_multiplier=0.3,
        num_sampled_clients=6,
    )

    # Adjust to 5 rounds to ensure DP guarantees hold
    config = ServerConfig(num_rounds=5)

    return ServerAppComponents(
        strategy=fedavg_with_dp,
        config=config,
    )

### **Create Server Application**


In [ ]:
server = ServerApp(server_fn=server_fn)


---

## **Step 5: Run Federated Training with Privacy**

In [ ]:
run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=10,
    backend_config=backend_setup,
)

DEBUG:flwr:Asyncio event loop already running.
INFO : Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO : 
INFO : [INIT]
INFO : Using initial global parameters provided by strategy
INFO : Evaluating initial global parameters
INFO : 
INFO : [ROUND 1]
INFO : configure_fit: strategy sampled 6 clients (out of 10)
(pid=2738) 2025-02-08 22:04:04.267739: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=2738) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=2738) E0000 00:00:1739052244.308198    2738 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=2738) E0000 00:00:1739052244.321382    2738 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one ha